In [1]:
import re
import ast
import pandas as pd

RAW_PATH = "../data/raw/resume_data_for_ranking.csv"
OUTPUT_PATH = "../data/processed/resume_data_clean.csv"

df = pd.read_csv(RAW_PATH)
print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Data loaded: 9544 rows, 35 columns


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667
3,NaN,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...","['Martinez Adult Education, Business Training ...",['Computer Applications Specialist Certificate...,['2008'],[None],[None],['Computer Applications'],"['Company Name ï¼ City , State', 'Company Name...",...,NaN,NaN,NaN,Business Development Executive,Bachelor/Honors,1 to 3 years,Age 22 to 30 years,Apparel Sourcing\nQuality Garment Sourcing\nRe...,Fast typing skill\nIELTSInternet browsing & on...,0.760000
4,NaN,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Kent State University'],['Bachelor of Business Administration'],[None],['3.84'],[None],['Accounting'],"['Company Name', 'Company Name', 'Company Name...",...,[None],[None],"['February 15, 2021']",Senior iOS Engineer,Bachelor of Science (BSc) in Computer Science,At least 4 years,NaN,iOS Lifecycle\nRequirement Analysis\nNative Fr...,iOS\niOS App Developer\niOS Application Develo...,0.650000


In [2]:
def clean_text_general(text):
    """Aggressive cleaning for free text intended for vectorization."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("\\n", " ").replace("\n", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # keeps letters/digits/spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_skill(skill):
    """Light cleaning for a single skill (preserves C++, C#, .NET, etc.)."""
    skill = str(skill).strip().lower()
    skill = re.sub(r"\s+", " ", skill)
    return skill

def parse_skills_normalized(x):
    """Parse the 'skills' column (stringified list) into a set of normalized skills."""
    try:
        items = ast.literal_eval(x)
        return set(normalize_skill(s) for s in items if str(s).strip())
    except Exception:
        return set()

def parse_required_normalized(x):
    """Parse 'skills_required' (multiline text) into a set of normalized skills."""
    if pd.isna(x):
        return set()
    return set(normalize_skill(s) for s in x.split("\n") if s.strip())

In [3]:
df["skills_set_clean"] = df["skills"].apply(parse_skills_normalized)
df["required_set_clean"] = df["skills_required"].apply(parse_required_normalized)
df["skills_overlap"] = df.apply(
    lambda r: len(r["skills_set_clean"] & r["required_set_clean"]), axis=1
)

df["skills_overlap"].value_counts().sort_index()

skills_overlap
0    9064
1     419
2      59
3       2
Name: count, dtype: int64

In [4]:
text_cols = [
    "career_objective",
    "responsibilities",
    "responsibilities.1",
    "educationaL_requirements",
]

for col in text_cols:
    df[col + "_clean"] = df[col].apply(clean_text_general)

df[["career_objective", "career_objective_clean"]].sample(3, random_state=42)

,career_objective,career_objective_clean
5470,NaN,
8982,NaN,
4704,NaN,


In [5]:
missing_flags = {
    "career_objective": "has_career_objective",
    "certification_providers": "has_certifications",
    "languages": "has_languages",
    "extra_curricular_activity_types": "has_extra_curricular",
    "skills_required": "has_skills_required",
    "experiencere_requirement": "has_experience_requirement",
    "age_requirement": "has_age_requirement",
}

for source_col, flag_col in missing_flags.items():
    df[flag_col] = df[source_col].notna().astype(int)

df[list(missing_flags.values())].mean().sort_values()

has_languages                 0.073345
has_certifications            0.210394
has_extra_curricular          0.358969
has_career_objective          0.496647
has_age_requirement           0.571773
has_skills_required           0.821773
has_experience_requirement    0.857083
dtype: float64

In [6]:
cols_to_drop = [
    "address",
    "languages",
    "proficiency_levels",
    "company_urls",
    "online_links",
    "extra_curricular_organization_links",
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Shape after dropping sparse columns : {df.shape}")

Shape after dropping sparse columns : (9544, 43)


In [7]:
print("NaNs remaining in cleaned text columns :")
print(df[[c + "_clean" for c in text_cols]].isna().sum())

print("\nPreview :")
df[["skills_overlap"] + list(missing_flags.values())].head()

NaNs remaining in cleaned text columns :
career_objective_clean            0
responsibilities_clean            0
responsibilities.1_clean          0
educationaL_requirements_clean    0
dtype: int64

Preview :


,skills_overlap,has_career_objective,has_certifications,has_languages,has_extra_curricular,has_skills_required,has_experience_requirement,has_age_requirement
0,0,1,0,0,0,0,1,0
1,0,1,0,0,0,0,1,0
2,0,0,0,0,0,1,1,0
3,0,1,0,0,0,1,1,1
4,0,1,1,0,1,1,1,0


In [8]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"File saved: {OUTPUT_PATH} ({df.shape[0]} rows, {df.shape[1]} columns)")

File saved: ../data/processed/resume_data_clean.csv (9544 rows, 43 columns)
